# 🚀 DBT (Data Build Tool) - Complete Detailed Notes

------------------------------------------------------------------------

## 📌 What is DBT?

DBT (Data Build Tool) is an open-source tool used for transforming data
inside a data warehouse using SQL.

-   Focus: **Transform (T) in ELT**
-   Works on top of:
    -   Snowflake
    -   BigQuery
    -   Redshift
    -   Databricks

------------------------------------------------------------------------

## 🧠 Core Philosophy

DBT brings **software engineering practices to data**:

-   Modular SQL
-   Version control (Git)
-   Testing
-   Documentation
-   Reusability

------------------------------------------------------------------------

## ⚙️ How DBT Works

1.  Raw data loaded into warehouse
2.  DBT models transform data
3.  DAG auto-created using dependencies
4.  Output tables used for analytics

------------------------------------------------------------------------

## 🧩 Key Concepts (Detailed)

### 1. Models

Each SQL file = a model

``` sql
-- models/staging/stg_orders.sql
SELECT
    id,
    amount,
    created_at
FROM raw.orders
```

------------------------------------------------------------------------

### 2. Materializations

``` sql
{{ config(materialized='table') }}
```

Types: - view - table - incremental

------------------------------------------------------------------------

### 3. Incremental Model (Production Use)

``` sql
{{ config(materialized='incremental') }}

SELECT *
FROM raw.orders

{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
{% endif %}
```

------------------------------------------------------------------------

### 4. DAG using ref()

``` sql
SELECT *
FROM {{ ref('stg_orders') }}
```

👉 Automatically creates dependency graph

------------------------------------------------------------------------

### 5. Tests

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - unique
          - not_null
```

------------------------------------------------------------------------

### 6. Macros

``` sql
{% macro tax(amount) %}
    amount * 0.18
{% endmacro %}
```

Usage:

``` sql
SELECT {{ tax('amount') }} FROM table
```

------------------------------------------------------------------------

### 7. Snapshots (SCD Type 2)

``` sql
{% snapshot orders_snapshot %}

{{
    config(
      target_schema='snapshots',
      unique_key='id',
      strategy='timestamp',
      updated_at='updated_at'
    )
}}

SELECT * FROM raw.orders

{% endsnapshot %}
```

------------------------------------------------------------------------

### 8. Documentation

``` bash
dbt docs generate
dbt docs serve
```

------------------------------------------------------------------------

## 🏗️ Project Structure

    models/
      staging/
      intermediate/
      marts/

    macros/
    tests/
    snapshots/

------------------------------------------------------------------------

## 🔥 Why DBT?

### 1. SQL First

-   Easy for analysts
-   No heavy coding

### 2. ELT Approach

-   Uses warehouse power

### 3. Version Control

-   Git-based workflow

### 4. Testing Layer

-   Ensures data quality

### 5. Modularity

-   Reusable SQL models

------------------------------------------------------------------------

## 📈 Why DBT is Trending

-   Rise of Modern Data Stack
-   Analytics Engineering role
-   Faster delivery
-   Lower infrastructure cost
-   Strong ecosystem

------------------------------------------------------------------------

## ⚠️ Limitations

-   No ingestion
-   SQL-heavy
-   Not ideal for ML
-   Depends on warehouse performance

------------------------------------------------------------------------

## 🏗️ Real Production-Level Example

### Staging Layer

``` sql
-- stg_orders.sql
SELECT
    id,
    customer_id,
    amount,
    created_at
FROM raw.orders
```

------------------------------------------------------------------------

### Intermediate Layer

``` sql
-- int_orders.sql
SELECT
    customer_id,
    COUNT(*) as total_orders,
    SUM(amount) as total_spent
FROM {{ ref('stg_orders') }}
GROUP BY customer_id
```

------------------------------------------------------------------------

### Mart Layer

``` sql
-- mart_customer_revenue.sql
SELECT
    customer_id,
    total_orders,
    total_spent,
    CASE
        WHEN total_spent > 10000 THEN 'High Value'
        ELSE 'Low Value'
    END as segment
FROM {{ ref('int_orders') }}
```

------------------------------------------------------------------------

## 🆚 DBT vs Traditional ETL

  Feature    ETL        DBT
  ---------- ---------- -----------
  Language   Python     SQL
  Engine     External   Warehouse
  Testing    Manual     Built-in
  Cost       High       Lower

------------------------------------------------------------------------

## 🎯 Summary

DBT: - Simplifies transformation - Enables analytics engineering -
Improves data quality - Speeds up development

------------------------------------------------------------------------

## 🚀 Next Steps

-   DBT + Airflow integration
-   DBT with Databricks
-   CI/CD with DBT
-   Advanced macros & packages
